# Tome C2, Chapter 4: Graph Algorithms (Python companion)

Exercises first, solutions at the end. We implement every algorithm of the chapter from scratch, on adjacency lists, and check each one against the `networkx` library on thousands of random graphs: breadth-first search, depth-first search with timestamps, both algorithms for strongly connected components, $2$-SAT (with a surprise: a phase transition), cut vertices and bridges, Dijkstra, Bellman–Ford, Floyd–Warshall and Johnson, an arbitrage hunt, and the three minimum spanning tree algorithms. We end in a maze, as the chapter began.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time, heapq, itertools
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
random.seed(4)

def random_digraph(n, m, seed=None):
    rng = random.Random(seed)
    m = min(m, n * (n - 1)); edges = set()
    while len(edges) < m:
        u, v = rng.randrange(n), rng.randrange(n)
        if u != v: edges.add((u, v))
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v)
    return adj

def random_graph(n, m, seed=None):
    rng = random.Random(seed)
    m = min(m, n * (n - 1) // 2); edges = set()
    while len(edges) < m:
        u, v = rng.randrange(n), rng.randrange(n)
        if u != v: edges.add((min(u, v), max(u, v)))
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    return adj, sorted(edges)

## Exercises

**P4.1 (★) Breadth-first search.** Implement BFS and check Theorem 4.2 against `networkx`. Use Corollary 4.3 to test bipartiteness, returning either a $2$-colouring or an odd cycle, and verify the certificate.

**P4.2 (★★) Depth-first search.** Implement DFS *without recursion* (Python's recursion limit is about $1000$), with discovery and finishing times and the classification of edges. Check the parenthesis theorem on random digraphs, and Theorem 4.6: a back edge appears exactly when the graph has a cycle, and decreasing finishing times give a topological order.

**P4.3 (★★) Strongly connected components.** Implement Kosaraju's and Tarjan's algorithms (iteratively), compare them with `networkx`, and time them to see linear growth. Watch the giant component appear in random digraphs as the number of edges grows past $n$.

**P4.4 (★★) 2-SAT.** Implement the solver of Theorem 4.10, verify its models, and compare with brute force on small instances. Then draw random $2$-CNFs with $m=cn$ clauses and plot the probability of satisfiability against $c$, for several $n$.

**P4.5 (★) Cut vertices and bridges.** Implement Theorem 4.12 and compare with `networkx`. Orient a bridgeless graph as in Exercise 4.8 and check that the result is strongly connected.

**P4.6 (★★) Shortest paths.** Implement Dijkstra (with the lazy heap of the remark after Theorem 4.14), Bellman–Ford, Floyd–Warshall and Johnson; check them against each other and against `networkx`; reproduce the failure of Exercise 4.4; time Dijkstra against Bellman–Ford.

**P4.7 (★) Arbitrage.** Generate exchange rates that are consistent up to small noise, find a profitable cycle with Bellman–Ford, and check its product.

**P4.8 (★) Minimum spanning trees.** Implement Kruskal, Prim and Borůvka, check that they agree with `networkx`, and count Borůvka's phases against $\log_2 n$.

**P4.9 (★) A maze.** Carve a random maze with a randomised depth-first search, solve it with BFS, and draw both.

## Solutions

### P4.1 Breadth-first search

In [ ]:
def bfs(adj, s):
    d = [None] * len(adj); par = [None] * len(adj)
    d[s] = 0; Q = deque([s])
    while Q:
        u = Q.popleft()
        for v in adj[u]:
            if d[v] is None:
                d[v] = d[u] + 1; par[v] = u; Q.append(v)
    return d, par

for trial in range(200):
    n = random.randrange(2, 60); m = random.randrange(0, min(n * (n - 1) // 2, 3 * n) + 1)
    adj, edges = random_graph(n, m, seed=trial)
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
    d, _ = bfs(adj, 0)
    ref = nx.single_source_shortest_path_length(G, 0)
    assert all(d[v] == ref.get(v) for v in range(n))
print("BFS distances agree with networkx on 200 random graphs")

def bipartite(adj):
    # returns ('colouring', colour list) or ('odd cycle', list of vertices)
    n = len(adj); d = [None] * n; par = [None] * n
    for s in range(n):
        if d[s] is not None: continue
        d[s] = 0; Q = deque([s])
        while Q:
            u = Q.popleft()
            for v in adj[u]:
                if d[v] is None:
                    d[v] = d[u] + 1; par[v] = u; Q.append(v)
                elif d[v] == d[u]:
                    # climb from u and v to their lowest common ancestor
                    pu, pv = [u], [v]
                    while pu[-1] != pv[-1]:
                        pu.append(par[pu[-1]]); pv.append(par[pv[-1]])
                    return 'odd cycle', pu + pv[-2::-1]
    return 'colouring', [x % 2 for x in d]

counts = {'colouring': 0, 'odd cycle': 0}
for trial in range(500):
    n = random.randrange(2, 40); m = random.randrange(0, 2 * n)
    adj, edges = random_graph(n, min(m, n * (n - 1) // 2), seed=1000 + trial)
    kind, cert = bipartite(adj)
    counts[kind] += 1
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
    assert (kind == 'colouring') == nx.is_bipartite(G)
    if kind == 'colouring':
        assert all(cert[u] != cert[v] for u, v in edges)
    else:
        assert len(cert) % 2 == 1 and len(set(cert)) == len(cert)
        assert all(G.has_edge(cert[i], cert[(i + 1) % len(cert)]) for i in range(len(cert)))
print("certificates checked:", counts)

### P4.2 Depth-first search
An explicit stack of iterators replaces the recursion: the top of the stack is the call in progress, and advancing its iterator is the loop over $\mathrm{Adj}(u)$.

In [ ]:
def dfs(adj, order=None):
    n = len(adj); d = [0] * n; f = [0] * n; par = [None] * n; kind = {}
    t = 0
    for s in (order if order is not None else range(n)):
        if d[s]: continue
        t += 1; d[s] = t; stack = [(s, iter(adj[s]))]
        while stack:
            u, it = stack[-1]
            for v in it:
                if not d[v]:
                    kind[(u, v)] = 'tree'; par[v] = u
                    t += 1; d[v] = t; stack.append((v, iter(adj[v])))
                    break
                elif not f[v]: kind[(u, v)] = 'back'
                elif d[u] < d[v]: kind[(u, v)] = 'forward'
                else: kind[(u, v)] = 'cross'
            else:
                stack.pop(); t += 1; f[u] = t
    return d, f, par, kind

# the graph of Figure 4.1
names = "abcdefgh"; A = {"a": "be", "b": "cd", "c": "a", "d": "e", "e": "f", "f": "dg", "g": "", "h": "eg"}
adj8 = [[names.index(y) for y in A[x]] for x in names]
d, f, par, kind = dfs(adj8)
print({names[v]: f"{d[v]}/{f[v]}" for v in range(8)})
print({names[u] + names[v]: k for (u, v), k in kind.items()})

def is_ancestor(par, u, v):
    while v is not None:
        if v == u: return True
        v = par[v]
    return False

for trial in range(300):
    n = random.randrange(2, 40); adj = random_digraph(n, random.randrange(0, 3 * n), seed=trial)
    d, f, par, kind = dfs(adj)
    for u in range(n):
        for v in range(n):
            if u == v: continue
            disjoint = f[u] < d[v] or f[v] < d[u]
            nested = d[u] < d[v] < f[v] < f[u]
            assert disjoint or nested or d[v] < d[u] < f[u] < f[v]
            assert nested == is_ancestor(par, u, v)
    has_back = any(k == 'back' for k in kind.values())
    G = nx.DiGraph(); G.add_nodes_from(range(n)); G.add_edges_from((u, v) for u in range(n) for v in adj[u])
    assert has_back == (not nx.is_directed_acyclic_graph(G))
    if not has_back:
        pos = {v: i for i, v in enumerate(sorted(range(n), key=lambda v: -f[v]))}
        assert all(pos[u] < pos[v] for u, v in G.edges)
print("parenthesis theorem, cycle test and topological order checked on 300 random digraphs")

### P4.3 Strongly connected components

In [ ]:
def kosaraju(adj):
    n = len(adj)
    _, f, _, _ = dfs(adj)
    radj = [[] for _ in range(n)]
    for u in range(n):
        for v in adj[u]: radj[v].append(u)
    comp = [None] * n; c = 0
    for s in sorted(range(n), key=lambda v: -f[v]):
        if comp[s] is not None: continue
        comp[s] = c; stack = [s]
        while stack:
            u = stack.pop()
            for v in radj[u]:
                if comp[v] is None: comp[v] = c; stack.append(v)
        c += 1
    return comp          # components numbered in a topological order of the condensation

def tarjan(adj):
    n = len(adj); d = [0] * n; low = [0] * n; onS = [False] * n; S = []
    comp = [None] * n; c = 0; t = 0
    for s in range(n):
        if d[s]: continue
        t += 1; d[s] = low[s] = t; S.append(s); onS[s] = True
        stack = [(s, iter(adj[s]))]
        while stack:
            u, it = stack[-1]
            for v in it:
                if not d[v]:
                    t += 1; d[v] = low[v] = t; S.append(v); onS[v] = True
                    stack.append((v, iter(adj[v]))); break
                elif onS[v]:
                    low[u] = min(low[u], d[v])
            else:
                stack.pop()
                if stack:
                    p = stack[-1][0]; low[p] = min(low[p], low[u])
                if low[u] == d[u]:
                    while True:
                        w = S.pop(); onS[w] = False; comp[w] = c
                        if w == u: break
                    c += 1
    return comp          # components numbered in reverse topological order

def same_partition(c1, c2):
    return len(set(c1)) == len(set(zip(c1, c2))) == len(set(c2))

for trial in range(300):
    n = random.randrange(1, 50); adj = random_digraph(n, random.randrange(0, 3 * n) if n > 1 else 0, seed=trial)
    G = nx.DiGraph(); G.add_nodes_from(range(n)); G.add_edges_from((u, v) for u in range(n) for v in adj[u])
    ref = [None] * n
    for i, C in enumerate(nx.strongly_connected_components(G)):
        for v in C: ref[v] = i
    k, t = kosaraju(adj), tarjan(adj)
    assert same_partition(k, ref) and same_partition(t, ref)
    assert all(k[u] <= k[v] for u in range(n) for v in adj[u])       # topological
    assert all(t[u] >= t[v] for u in range(n) for v in adj[u])       # reverse topological
print("Kosaraju and Tarjan agree with networkx on 300 random digraphs, with the orders claimed")

ns = [20000, 40000, 80000, 160000]; T = {'Kosaraju': [], 'Tarjan': []}
for n in ns:
    adj = random_digraph(n, 3 * n, seed=n)
    for name, fn in (('Kosaraju', kosaraju), ('Tarjan', tarjan)):
        t0 = time.perf_counter(); fn(adj); T[name].append(time.perf_counter() - t0)
for name in T: print(name, ["%.2f s" % x for x in T[name]], " time / (n+m) in microseconds:", ["%.2f" % (1e6 * x / (4 * n)) for x, n in zip(T[name], ns)])

Both are linear in principle; in practice the time per vertex-plus-edge creeps up with $n$ because large graphs no longer fit in the processor's caches, and memory, not arithmetic, is the bottleneck. Tarjan's single pass is visibly cheaper. Now the giant component. For a random digraph with $m=cn$ edges, the largest strongly connected component stays tiny while $c<1$ and then occupies a positive fraction of the vertices (Karp, 1990). Compare with the giant component of random graphs in Tome C1.

In [ ]:
n = 4000; cs = np.linspace(0.2, 3, 29); frac = []
for c in cs:
    comp = tarjan(random_digraph(n, int(c * n), seed=int(100 * c)))
    frac.append(max(np.bincount(comp)) / n)
plt.figure(figsize=(6, 3.2))
plt.plot(cs, frac, 'o-', color='tab:blue')
plt.axvline(1, color='gray', ls='--')
plt.xlabel("edges per vertex $c=m/n$"); plt.ylabel("largest SCC / n"); plt.title(f"random digraphs, n={n}")
plt.tight_layout(); plt.show()

### P4.4 2-SAT
A literal is an integer: $2i$ for $x_i$ and $2i+1$ for $\neg x_i$, so negation is `l ^ 1`.

In [ ]:
def two_sat(n, clauses):
    adj = [[] for _ in range(2 * n)]
    for a, b in clauses:
        adj[a ^ 1].append(b); adj[b ^ 1].append(a)
    comp = kosaraju(adj)                      # topological numbering
    if any(comp[2 * i] == comp[2 * i + 1] for i in range(n)): return None
    return [comp[2 * i] > comp[2 * i + 1] for i in range(n)]   # x true iff t(x) > t(not x)

def satisfies(model, clauses):
    val = lambda l: model[l >> 1] ^ (l & 1)
    return all(val(a) or val(b) for a, b in clauses)

def random_2cnf(n, m, rng):
    return [tuple(2 * rng.randrange(n) + rng.randrange(2) for _ in range(2)) for _ in range(m)]

rng = random.Random(7)
for trial in range(2000):
    n = rng.randrange(1, 9); cl = random_2cnf(n, rng.randrange(1, 3 * n + 2), rng)
    model = two_sat(n, cl)
    brute = any(satisfies(list(bits), cl) for bits in itertools.product([False, True], repeat=n))
    assert (model is not None) == brute
    if model is not None: assert satisfies(model, cl)
print("2-SAT solver agrees with brute force on 2000 small formulas, and its models are checked")

Now the surprise. With $m=cn$ random clauses, a random $2$-CNF is almost surely satisfiable when $c<1$ and almost surely unsatisfiable when $c>1$, as $n\to\infty$: a sharp threshold, proved independently by Chvátal and Reed (1992) and by Goerdt (1996). Watch the curve steepen with $n$.

In [ ]:
cs = np.linspace(0.3, 2.0, 18)
plt.figure(figsize=(6, 3.4))
for n, trials in ((50, 300), (500, 200), (5000, 60)):
    p = [np.mean([two_sat(n, random_2cnf(n, int(c * n), rng)) is not None for _ in range(trials)]) for c in cs]
    plt.plot(cs, p, 'o-', ms=3, label=f"n={n}")
plt.axvline(1, color='gray', ls='--'); plt.xlabel("clauses per variable $c$"); plt.ylabel("P(satisfiable)")
plt.legend(); plt.tight_layout(); plt.show()

### P4.5 Cut vertices and bridges

In [ ]:
def cut_vertices_and_bridges(adj):
    n = len(adj); d = [0] * n; low = [0] * n; par = [None] * n; t = 0
    cuts, bridges = set(), []
    for s in range(n):
        if d[s]: continue
        t += 1; d[s] = low[s] = t; stack = [(s, iter(adj[s]))]; root_children = 0
        while stack:
            u, it = stack[-1]
            for v in it:
                if not d[v]:
                    par[v] = u; t += 1; d[v] = low[v] = t; stack.append((v, iter(adj[v])))
                    if u == s: root_children += 1
                    break
                elif v != par[u]:
                    low[u] = min(low[u], d[v])          # back edge (graph is simple)
            else:
                stack.pop()
                if stack:
                    p = stack[-1][0]; low[p] = min(low[p], low[u])
                    if p != s and low[u] >= d[p]: cuts.add(p)
                    if low[u] > d[p]: bridges.append((min(p, u), max(p, u)))
        if root_children >= 2: cuts.add(s)
    return cuts, sorted(bridges)

for trial in range(300):
    n = random.randrange(2, 50); m = random.randrange(n - 1, 2 * n)
    adj, edges = random_graph(n, min(m, n * (n - 1) // 2), seed=5000 + trial)
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
    cuts, br = cut_vertices_and_bridges(adj)
    assert cuts == set(nx.articulation_points(G))
    assert br == sorted((min(u, v), max(u, v)) for u, v in nx.bridges(G))
print("cut vertices and bridges agree with networkx on 300 random graphs")

def robbins_orientation(adj):
    d, f, par, kind = dfs(adj)          # undirected: each edge scanned from both ends
    arcs = []
    for (u, v), k in kind.items():
        if k == 'tree': arcs.append((u, v))                 # parent to child
        elif k == 'back' and par[u] != v: arcs.append((u, v))   # descendant to ancestor
    return arcs

ok = 0
for trial in range(300):
    n = random.randrange(3, 40)
    adj, edges = random_graph(n, min(2 * n, n * (n - 1) // 2), seed=9000 + trial)
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
    if not nx.is_connected(G) or any(True for _ in nx.bridges(G)): continue
    arcs = robbins_orientation(adj)
    assert len(arcs) == len(edges)
    D = nx.DiGraph(arcs); assert nx.is_strongly_connected(D); ok += 1
print(f"Robbins orientations strongly connected on all {ok} bridgeless connected samples")

### P4.6 Shortest paths

In [ ]:
INF = float('inf')

def dijkstra(wadj, s):
    n = len(wadj); D = [INF] * n; D[s] = 0; done = [False] * n; H = [(0, s)]
    while H:
        du, u = heapq.heappop(H)
        if done[u]: continue                  # stale entry
        done[u] = True
        for v, w in wadj[u]:
            if du + w < D[v]:
                D[v] = du + w; heapq.heappush(H, (D[v], v))
    return D

def bellman_ford(wadj, s):
    n = len(wadj); D = [INF] * n; D[s] = 0
    bellman_ford.rounds = 0
    for _ in range(n - 1):
        changed = False; bellman_ford.rounds += 1
        for u in range(n):
            if D[u] == INF: continue
            for v, w in wadj[u]:
                if D[u] + w < D[v]: D[v] = D[u] + w; changed = True
        if not changed: break
    for u in range(n):
        if D[u] < INF and any(D[u] + w < D[v] for v, w in wadj[u]): return None   # negative cycle
    return D

def floyd_warshall(n, wedges):
    D = np.full((n, n), np.inf); np.fill_diagonal(D, 0)
    for u, v, w in wedges: D[u, v] = min(D[u, v], w)
    for k in range(n):
        D = np.minimum(D, D[:, [k]] + D[[k], :])     # one round of the recurrence, vectorised
    return D

def johnson(wadj):
    n = len(wadj)
    h = bellman_ford(wadj + [[(v, 0) for v in range(n)]], n)
    if h is None: return None
    rew = [[(v, w + h[u] - h[v]) for v, w in wadj[u]] for u in range(n)]
    assert all(w >= -1e-9 for row in rew for _, w in row)
    return [[Dv - h[u] + h[v] if Dv < INF else INF for v, Dv in enumerate(dijkstra(rew, u))] for u in range(n)]

def random_weighted(n, m, lo, hi, seed):
    adj = random_digraph(n, m, seed); rng = random.Random(seed)
    wadj = [[(v, rng.randint(lo, hi)) for v in adj[u]] for u in range(n)]
    return wadj, [(u, v, w) for u in range(n) for v, w in wadj[u]]

for trial in range(150):
    n = random.randrange(2, 30); wadj, we = random_weighted(n, random.randrange(0, 4 * n), 0, 20, trial)
    G = nx.DiGraph(); G.add_nodes_from(range(n)); G.add_weighted_edges_from(we)
    ref = nx.single_source_dijkstra_path_length(G, 0)
    D = dijkstra(wadj, 0)
    assert all(D[v] == ref.get(v, INF) for v in range(n))
    assert D == bellman_ford(wadj, 0)
neg = 0
for trial in range(300):
    n = random.randrange(2, 25); wadj, we = random_weighted(n, random.randrange(0, 3 * n), -4, 20, 777 + trial)
    G = nx.DiGraph(); G.add_nodes_from(range(n)); G.add_weighted_edges_from(we)
    J = johnson(wadj); F = floyd_warshall(n, we)
    has_neg = nx.negative_edge_cycle(G)
    assert (J is None) == has_neg == bool((np.diag(F) < 0).any())     # Exercise 4.9
    if J is None: neg += 1; continue
    assert np.array_equal(np.array(J, dtype=float), F)
    ref = dict(nx.all_pairs_bellman_ford_path_length(G))
    assert all(F[u, v] == ref[u].get(v, INF) for u in range(n) for v in range(n))
print(f"Dijkstra, Bellman-Ford, Floyd-Warshall and Johnson agree with networkx; {neg} of 300 graphs had a negative cycle, all detected")

# Exercise 4.4: the three-vertex counterexample (s=0, a=1, b=2)
bad = [[(1, 2), (2, 3)], [], [(1, -2)]]
def dijkstra_settled(wadj, s):
    n = len(wadj); D = [INF] * n; D[s] = 0; done = [False] * n; H = [(0, s)]; settled = {}
    while H:
        du, u = heapq.heappop(H)
        if done[u]: continue
        done[u] = True; settled[u] = du
        for v, w in wadj[u]:
            if not done[v] and du + w < D[v]: D[v] = du + w; heapq.heappush(H, (D[v], v))
    return settled
print("Dijkstra settles a at", dijkstra_settled(bad, 0)[1], "but the true distance is", bellman_ford(bad, 0)[1])

In [ ]:
ns = [500, 1000, 2000, 4000]; td, tb, rounds = [], [], []
for n in ns:
    wadj, _ = random_weighted(n, 8 * n, 1, 100, n)
    t0 = time.perf_counter(); dijkstra(wadj, 0); td.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); bellman_ford(wadj, 0); tb.append(time.perf_counter() - t0); rounds.append(bellman_ford.rounds)
for n, a, b, r in zip(ns, td, tb, rounds): print(f"n={n:5d}, m=8n: Dijkstra {1000*a:7.1f} ms   Bellman-Ford {1000*b:8.1f} ms after {r} rounds (out of n-1)")

Our Bellman–Ford stops as soon as a round changes nothing, which on these random graphs happens after a handful of rounds, so the gap is far smaller than the worst-case ratio $n/\log n$. Exercise for the curious: build a graph that forces all $n-1$ rounds, for the order in which your code scans the edges.

### P4.7 Arbitrage
Rates are $r_{ij}=e^{p_j-p_i}$ for hidden "true prices" $p$, minus a fee of $0.2\%$ per exchange, times a small random noise. Without noise every cycle loses money; once the noise is comparable with the fees, opportunities appear, mostly on two- and three-currency cycles. We run Bellman–Ford from an added source on the weights $-\log r_{ij}$, keep the predecessor of each vertex, and when an edge can still be relaxed after the rounds we walk back $n$ predecessors (to be sure to land on a cycle) and read the cycle off.

In [ ]:
def find_negative_cycle(n, wedges):
    D = [0.0] * n; pred = [None] * n          # source q joined to all with weight 0: D starts at 0
    x = None
    for _ in range(n):
        x = None
        for u, v, w in wedges:
            if D[u] + w < D[v] - 1e-12: D[v] = D[u] + w; pred[v] = u; x = v
        if x is None: return None
    for _ in range(n): x = pred[x]
    cyc = [x]; y = pred[x]
    while y != x: cyc.append(y); y = pred[y]
    return cyc[::-1]

rng = np.random.default_rng(11); n = 12
p = rng.normal(0, 1, n)
fee = 0.002
for noise in (0.0, 0.0005, 0.001, 0.002, 0.004):
    found = 0
    for rep in range(200):
        R = np.exp(p[None, :] - p[:, None]) * np.exp(-fee + rng.normal(0, noise, (n, n)))
        we = [(i, j, -math.log(R[i, j])) for i in range(n) for j in range(n) if i != j]
        cyc = find_negative_cycle(n, we)
        if cyc:
            prod = np.prod([R[cyc[k], cyc[(k + 1) % len(cyc)]] for k in range(len(cyc))])
            assert prod > 1
            found += 1
    print(f"noise {noise}: arbitrage found in {found} of 200 markets" + (f"; last cycle {cyc} multiplies money by {prod:.5f}" if found and cyc else ""))

### P4.8 Minimum spanning trees

In [ ]:
class DSU:
    def __init__(self, n): self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x: self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b: return False
        if self.r[a] < self.r[b]: a, b = b, a
        self.p[b] = a; self.r[a] += self.r[a] == self.r[b]; return True

def kruskal(n, wedges):
    U = DSU(n); return [e for e in sorted(wedges, key=lambda e: e[2]) if U.union(e[0], e[1])]

def prim(n, wadj):
    inT = [False] * n; H = [(0, 0, -1)]; T = []
    while H:
        w, v, u = heapq.heappop(H)
        if inT[v]: continue
        inT[v] = True
        if u >= 0: T.append((u, v, w))
        for x, wx in wadj[v]:
            if not inT[x]: heapq.heappush(H, (wx, x, v))
    return T

def boruvka(n, wedges):
    U = DSU(n); T = []; phases = 0
    key = lambda i: (wedges[i][2], i)              # ties broken by edge number
    while len(T) < n - 1:
        best = {}
        for i, (u, v, w) in enumerate(wedges):
            a, b = U.find(u), U.find(v)
            if a == b: continue
            for c in (a, b):
                if c not in best or key(i) < key(best[c]): best[c] = i
        for i in set(best.values()):
            u, v, w = wedges[i]
            if U.union(u, v): T.append(wedges[i])
        phases += 1
    return T, phases

for trial in range(100):
    n = random.randrange(2, 200); adj, edges = random_graph(n, min(4 * n, n * (n - 1) // 2), seed=trial)
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
    if not nx.is_connected(G): continue
    rng = random.Random(trial); wedges = [(u, v, rng.randint(1, 50)) for u, v in edges]
    wadj = [[] for _ in range(n)]
    for u, v, w in wedges: wadj[u].append((v, w)); wadj[v].append((u, w))
    G = nx.Graph(); G.add_weighted_edges_from(wedges)
    ref = nx.minimum_spanning_tree(G).size(weight='weight')
    weights = [sum(w for *_, w in T) for T in (kruskal(n, wedges), prim(n, wadj), boruvka(n, wedges)[0])]
    assert weights == [ref] * 3
print("Kruskal, Prim and Boruvka find spanning trees of the same, minimum, weight as networkx")
for n in (1000, 10000, 100000):
    adj, edges = random_graph(n, 3 * n, seed=n)
    rng = random.Random(n); wedges = [(u, v, rng.random()) for u, v in edges] + [(i, i + 1, 2.0) for i in range(n - 1)]
    print(f"n={n:6d}: Boruvka used {boruvka(n, wedges)[1]} phases; log2 n = {math.log2(n):.1f}")

In practice each component's cheapest edge often meets another component's, and the number of components falls much faster than by half: the bound $\log_2 n$ is a worst case.

### P4.9 A maze
A randomised depth-first search on a grid, knocking down the wall to each newly discovered cell, carves a *perfect* maze: the passages form a spanning tree (the DFS tree), so there is exactly one route between any two cells. Tremaux would have been at home. BFS then finds the route.

In [ ]:
W, H = 30, 18; rng = random.Random(1882)
cell = lambda x, y: y * W + x
passages = [[] for _ in range(W * H)]
seen = [False] * (W * H); seen[0] = True; stack = [(0, 0)]
while stack:
    x, y = stack[-1]
    nbrs = [(x + dx, y + dy) for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1))
            if 0 <= x + dx < W and 0 <= y + dy < H and not seen[cell(x + dx, y + dy)]]
    if not nbrs: stack.pop(); continue
    nx_, ny_ = rng.choice(nbrs); seen[cell(nx_, ny_)] = True
    passages[cell(x, y)].append(cell(nx_, ny_)); passages[cell(nx_, ny_)].append(cell(x, y))
    stack.append((nx_, ny_))
assert sum(map(len, passages)) // 2 == W * H - 1          # a spanning tree
d, par = bfs(passages, 0)
route = [W * H - 1]
while route[-1] != 0: route.append(par[route[-1]])

fig, ax = plt.subplots(figsize=(8, 4.9))
for y in range(H):
    for x in range(W):
        c = cell(x, y)
        if x + 1 >= W or cell(x + 1, y) not in passages[c]: ax.plot([x + 1, x + 1], [y, y + 1], 'k', lw=1)
        if y + 1 >= H or cell(x, y + 1) not in passages[c]: ax.plot([x, x + 1], [y + 1, y + 1], 'k', lw=1)
ax.plot([0, 0], [0, H], 'k', lw=1); ax.plot([0, W], [0, 0], 'k', lw=1)
ax.plot([c % W + 0.5 for c in route], [c // W + 0.5 for c in route], color='tab:orange', lw=2)
ax.set_aspect('equal'); ax.axis('off'); ax.set_title(f"route of {d[W*H-1]} steps through a {W}x{H} maze")
plt.show()